# Analýza členské základny ČAVO

Česká asociace pro vzácná onemocnění (ČAVO) sdružuje individuální členy (pacienty se vzácným onemocněním, kteří nemají pro svou diagnózu pacientskou organizaci) a pacientské organizace (zastupující jedno či více vzácných onemocnění). Cílem této analýzy je zmapovat strukturu členské základny, geografické rozložení členů a vývoj členství v čase.

**Zdroj dat:** Interní databáze ČAVO (data k 30. 4. 2026)  
**Data byla před analýzou anonymizována** – odstraněny veškeré osobní identifikátory (jména, kontakty, adresy). 
Analýza pracuje výhradně s agregovanými údaji. Pro účely analýzy se počítalo s tím, že každá pacientská organizace zastupuje jednu diagnózu.

## 1. Načtení a čištění dat

Prvním krokem bylo vytvoření pracovních kopií obou databází a odstranění sloupců obsahujících osobní identifikátory. 
Data byla exportována z interní databáze do formátu CSV. 
Po načtení bylo nutné odstranit prázdné sloupce vzniklé exportem z Excelu a řádky bez přiřazeného ID.

In [3]:
import pandas as pd

# načtení dat
ind = pd.read_csv(r'C:\Users\cavo-lenovo21\Downloads\MG\DA_projekt\cavo_clenove_da\data\CAVO_IND.csv', sep=';')
org = pd.read_csv(r'C:\Users\cavo-lenovo21\Downloads\MG\DA_projekt\cavo_clenove_da\data\CAVO_ORG.csv', sep=';')

# odstranění prázdných sloupců
ind = ind.dropna(axis=1, how='all')
org = org.dropna(axis=1, how='all')

print("Individuální členové:", ind.shape)
print("Organizace:", org.shape)
print("\nSloupce IND:", ind.columns.tolist())
print("\nSloupce ORG:", org.columns.tolist())

Individuální členové: (278, 9)
Organizace: (57, 10)

Sloupce IND: ['id', 'diagnoza', 'ERN', 'uv', 'prevalence', 'orphakod', 'mesto', 'rok_narozeni', 'rok_vzniku_clenstvi']

Sloupce ORG: ['id', 'nazev', 'diagnoza', 'ERN', 'prevalence', 'orphakod', 'mesto', 'kraj', 'rok_vzniku_org', 'rok_vzniku_clenstvi']


In [4]:
# odstranění řádků, kde není ID
ind = ind.dropna(subset=['id'])
org = org.dropna(subset=['id'])

print("Individuální členové:", len(ind))
print("Organizace:", len(org))

Individuální členové: 278
Organizace: 57


## 2. Přiřazení krajů podle PSČ

Individuální členové měli v databázi uloženo město a PSČ v jednom sloupci (formát `534 01 Holice`). Pro geografickou analýzu bylo nutné:
1. Extrahovat PSČ regulárním výrazem
2. Namapovat PSČ na kraj podle číselníku českých poštovních směrovacích čísel
3. Ručně doplnit kraj u záznamů bez PSČ (ve zdrojové databázi)

In [5]:
# Zkontroluj formát PSČ u individuálních členů
print(ind['mesto'].head(10))

0          534 01  Holice
1      566 01 Vysoké Mýto
2    348 02 Bor u Tachova
3         190 16  Praha 9
4        530 06 Pardubice
5          156 00 Praha 5
6    407 55 Dolní Podluží
7          739 94 Vedryně
8         199 00 Praha 16
9          198 00 Praha 9
Name: mesto, dtype: str


In [6]:
import re

# vytvoř sloupec psč
ind['psc'] = ind['mesto'].str.extract(r'(\d{3}\s?\d{2})')
ind['psc'] = ind['psc'].str.replace(' ', '')

# vytoření funkce pro přiřazení názvu kraje podle PSČ
def psc_na_kraj(psc):
    if pd.isna(psc):
        return None
    
    psc = str(psc).strip()
    prvni = psc[0]
    druhe = psc[:2]
    
    if prvni == '1':
        return 'Praha'
    elif prvni == '2':
        return 'Středočeský'
    elif druhe in ['30', '31', '32', '33', '34']:
        return 'Plzeňský'
    elif druhe in ['37', '38', '39']:
        return 'Jihočeský'
    elif druhe in ['35', '36']:
        return 'Karlovarský'
    elif druhe in ['40', '41', '42', '43', '44']:
        return 'Ústecký'
    elif druhe in ['46', '47']:
        return 'Liberecký'
    elif druhe in ['50', '51', '54']:
        return 'Královéhradecký'
    elif druhe in ['53', '56', '57']:
        return 'Pardubický'
    elif druhe in ['58', '59']:
        return 'Vysočina'
    elif prvni == '6':
        return 'Jihomoravský'
    elif druhe in ['75', '79']:
        return 'Olomoucký'
    elif druhe in ['70', '71', '72', '73', '74']:
        return 'Moravskoslezský'
    elif druhe in ['76', '77', '78']:
        return 'Zlínský'
    else:
        return None

ind['kraj'] = ind['psc'].apply(psc_na_kraj)

print(ind[['mesto', 'psc', 'kraj']].head(10))
print("\nChybějící kraj:", ind['kraj'].isna().sum())

                  mesto    psc             kraj
0        534 01  Holice  53401       Pardubický
1    566 01 Vysoké Mýto  56601       Pardubický
2  348 02 Bor u Tachova  34802         Plzeňský
3       190 16  Praha 9  19016            Praha
4      530 06 Pardubice  53006       Pardubický
5        156 00 Praha 5  15600            Praha
6  407 55 Dolní Podluží  40755          Ústecký
7        739 94 Vedryně  73994  Moravskoslezský
8       199 00 Praha 16  19900            Praha
9        198 00 Praha 9  19800            Praha

Chybějící kraj: 4


In [7]:
# Zkontroluj která PSČ chybí
chybejici = ind[ind['kraj'].isna()][['mesto', 'psc']]
print(chybejici)

    mesto  psc
114   NaN  NaN
136   NaN  NaN
186   NaN  NaN
197   NaN  NaN


## 3. Přehled vyčištěných dat

Po vyčištění obsahuje dataset 278 individuálních členů a 57 pacientských organizací. 
U 4 individuálních členů se nepodařilo přiřadit kraj z důvodu chybějícího města i PSČ.

In [8]:
# Přehled vyčištěných dat
print("=== INDIVIDUÁLNÍ ČLENOVÉ ===")
print(f"Celkem: {len(ind)}")
print(f"Chybějící kraj: {ind['kraj'].isna().sum()}")
print(f"\nRozdělení podle kraje:")
print(ind['kraj'].value_counts())

print("\n=== ORGANIZACE ===")
print(f"Celkem: {len(org)}")
print(f"\nRozdělení podle kraje:")
print(org['kraj'].value_counts())

=== INDIVIDUÁLNÍ ČLENOVÉ ===
Celkem: 278
Chybějící kraj: 4

Rozdělení podle kraje:
kraj
Praha              62
Středočeský        34
Jihomoravský       28
Moravskoslezský    24
Pardubický         21
Ústecký            20
Jihočeský          15
Plzeňský           13
Královéhradecký    13
Zlínský            12
Liberecký          11
Vysočina            8
Olomoucký           8
Karlovarský         5
Name: count, dtype: int64

=== ORGANIZACE ===
Celkem: 57

Rozdělení podle kraje:
kraj
Praha              30
Středočeský        10
Jihomoravský        4
Královéhradecký     3
Olomoucký           2
Karlovarský         2
Ústecký             2
Moravskoslezský     1
Plzeňský            1
Jihočeský           1
Name: count, dtype: int64


## 4. Kontrola dat před vizualizací

Závěrečná kontrola datových rozsahů před exportem do Power BI.

In [9]:
# Základní přehled dat před vizualizacemi
print("=== KONTROLA DAT ===")
print(f"\nIndividuální členové: {len(ind)}")
print(f"Organizace: {len(org)}")

print(f"\nRok vzniku členství IND - rozsah: {ind['rok_vzniku_clenstvi'].min()} - {ind['rok_vzniku_clenstvi'].max()}")
print(f"Rok vzniku členství ORG - rozsah: {org['rok_vzniku_clenstvi'].min()} - {org['rok_vzniku_clenstvi'].max()}")

print(f"\nRok narození IND - rozsah: {ind['rok_narozeni'].min()} - {ind['rok_narozeni'].max()}")
print(f"Chybějící rok narození: {ind['rok_narozeni'].isna().sum()}")

print(f"\nTop 5 diagnóz IND:")
print(ind['diagnoza'].value_counts().head())

=== KONTROLA DAT ===

Individuální členové: 278
Organizace: 57

Rok vzniku členství IND - rozsah: 2012.0 - 2026.0
Rok vzniku členství ORG - rozsah: 2012.0 - 2026.0

Rok narození IND - rozsah: 1932.0 - 2022.0
Chybějící rok narození: 7

Top 5 diagnóz IND:
diagnoza
ATTRV122I amyloidóza (Transthyretin-vázaná familiární amyloidní kardiomyopatie); ATTR-CM    8
Klasická forma mycosis fungoides, Primární kožní T-buněčný lymfom                           6
Syndrom fragilního X                                                                        5
McCuneův-Albrightův syndrom                                                                 4
Familiární středozemská horečka                                                             4
Name: count, dtype: int64


In [10]:
# Export vyčištěných dat pro Power BI
ind.to_csv(r'C:\Users\cavo-lenovo21\Downloads\MG\DA_projekt\cavo_clenove_da\data\CAVO_IND_clean.csv', index=False, encoding='utf-8-sig')
org.to_csv(r'C:\Users\cavo-lenovo21\Downloads\MG\DA_projekt\cavo_clenove_da\data\CAVO_ORG_clean.csv', index=False, encoding='utf-8-sig')

print("\nSloupce IND:", ind.columns.tolist())
print("Sloupce ORG:", org.columns.tolist())


Sloupce IND: ['id', 'diagnoza', 'ERN', 'uv', 'prevalence', 'orphakod', 'mesto', 'rok_narozeni', 'rok_vzniku_clenstvi', 'psc', 'kraj']
Sloupce ORG: ['id', 'nazev', 'diagnoza', 'ERN', 'prevalence', 'orphakod', 'mesto', 'kraj', 'rok_vzniku_org', 'rok_vzniku_clenstvi']


## 5. Vizualizace a závěry

Vyčištěná data byla exportována do Power BI, kde byly provedeny další transformace a následně vytvořen interaktivní dashboard. 
Dashboard je dostupný jako PDF v složce `/dashboard`.

**Hlavní zjištění:**
- Členská základna tvoří 278 individuálních členů a 57 pacientských organizací zastupujících 273 různých diagnóz
- Největší koncentrace členů je v Praze a Středočeském kraji
- Členství výrazně rostlo po roce 2020
- Nejvíce zastoupená věková skupina jsou členové ve věku 30–44 let